In [1]:
# Appraisal Analysis Gradio App (Regression)
# ------------------------------------------------

import pandas as pd
import numpy as np
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# -----------------------------
# Load Dataset
# -----------------------------

FILE_PATH = "appraisal_dataset.csv"
df = pd.read_csv(FILE_PATH)

required_cols = ["Rating", "Behavior", "Experience", "Band", "Band_Num", "Appraisal"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# -----------------------------
# Data Preparation
# -----------------------------

X = df.drop("Appraisal", axis=1)
y = df["Appraisal"]

categorical_cols = ["Behavior", "Band"]
numerical_cols = ["Rating", "Experience", "Band_Num"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numerical_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train
model.fit(X_train, y_train)

# Predictions
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# Evaluation
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

# -----------------------------
# EDA Functions
# -----------------------------

def eda_summary():
    return df.describe().to_string()


def eda_corr():
    return df.corr(numeric_only=True).to_string()

# -----------------------------
# Model Evaluation
# -----------------------------

def model_results():
    result = f"Train RMSE: {train_rmse:.2f}\n"
    result += f"Test RMSE: {test_rmse:.2f}\n\n"
    result += f"Train R2: {train_r2:.3f}\n"
    result += f"Test R2: {test_r2:.3f}\n"
    return result

# -----------------------------
# Model Correction (Simple Insight)
# -----------------------------

def model_correction():
    msg = "Model Diagnostics:\n"

    if test_r2 < 0.6:
        msg += "- Low R2: Model underfitting. Try adding features or non-linear models.\n"
    if train_r2 - test_r2 > 0.2:
        msg += "- Possible Overfitting: Consider regularization (Ridge/Lasso).\n"
    if test_rmse > train_rmse:
        msg += "- Generalization gap detected. Tune model.\n"

    msg += "\nRecommendations:\n"
    msg += "- Try Ridge/Lasso Regression\n"
    msg += "- Feature Engineering (interaction terms)\n"
    msg += "- Hyperparameter tuning\n"

    return msg

# -----------------------------
# Prediction Function
# -----------------------------

def predict(rating, behavior, experience, band, band_num):
    input_df = pd.DataFrame({
        "Rating": [rating],
        "Behavior": [behavior],
        "Experience": [experience],
        "Band": [band],
        "Band_Num": [band_num]
    })

    pred = model.predict(input_df)[0]
    return f"Predicted Appraisal: {pred:.2f}"

# -----------------------------
# Gradio UI
# -----------------------------

with gr.Blocks(title="Appraisal Analysis") as app:
    gr.Markdown("# Appraisal Analysis Dashboard")

    # EDA Tab
    with gr.Tab("EDA"):
        gr.Markdown("## Dataset Overview")
        gr.Dataframe(df.head())

        btn1 = gr.Button("Summary Stats")
        out1 = gr.Textbox()
        btn1.click(eda_summary, outputs=out1)

        btn2 = gr.Button("Correlation Matrix")
        out2 = gr.Textbox()
        btn2.click(eda_corr, outputs=out2)

    # Model Tab
    with gr.Tab("Model Evaluation"):
        gr.Markdown("## Regression Metrics")

        btn3 = gr.Button("Show Metrics")
        out3 = gr.Textbox()
        btn3.click(model_results, outputs=out3)

        btn4 = gr.Button("Model Correction Insights")
        out4 = gr.Textbox()
        btn4.click(model_correction, outputs=out4)

    # Prediction Tab
    with gr.Tab("Prediction"):
        gr.Markdown("## Predict Appraisal")

        rating_in = gr.Number(label="Rating")
        behavior_in = gr.Dropdown(choices=sorted(df["Behavior"].unique().tolist()), label="Behavior")
        experience_in = gr.Number(label="Experience")
        band_in = gr.Dropdown(choices=sorted(df["Band"].unique().tolist()), label="Band")
        band_num_in = gr.Number(label="Band_Num")

        btn5 = gr.Button("Predict")
        out5 = gr.Textbox()

        btn5.click(
            predict,
            inputs=[rating_in, behavior_in, experience_in, band_in, band_num_in],
            outputs=out5
        )

# Launch
if __name__ == "__main__":
    app.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
